In [24]:
using DataFrames, Distributions, CSV, Turing, Optim, StatsBase, StatsFuns, Random

Random.seed!(1234)

# Read the vitamin D dataset
data = CSV.read("../../data/vitd/nhanes.csv", DataFrame)
data[!, :"RIAGENDR"] = data[!, :"RIAGENDR"] .- 1
#data[!, :"DPQ020"] = data[!, :"DPQ020"] .+ 1
data[!, :"DPQ020"] = data[!, :"DPQ020"]

# Look at the first few rows of the data
first(data, 5)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,27.8,0.0,62.0,76.1,4.14,0.0,3.0
2,30.8,0.0,53.0,56.5,0.0,0.0,1.0
3,28.8,0.0,78.0,87.5,1.81,0.0,3.0
4,28.0,0.0,22.0,47.2,2.98,0.0,2.0
5,27.6,0.0,46.0,44.5,1.73,0.0,1.0


In [25]:
#= Convert categorical data to integers =#

data[!, :"RIAGENDR"] = convert(Array{Int}, data[!, :"RIAGENDR"])
data[!, :"DPQ020"] = convert(Array{Int}, data[!, "DPQ020"])
data[!, :"SMQ040"] = convert(Array{Int}, data[!, :"SMQ040"]);

# Get variables
bmi = data[!, :"BMXBMI"]
gender = data[!, :"RIAGENDR"]
age = data[!, :"RIDAGEYR"]
vitd = data[!, :"LBXVIDMS"]
poverty = data[!, :"INDFMMPI"]
depression = data[!, :"DPQ020"]
smoking = data[!, :"SMQ040"];
replace!(data.:"DPQ020", 0 => 0, 1 => 0, 2 => 1, 3 => 1);

In [26]:
poverty_array = convert(Vector{Float64}, poverty);

In [27]:
# Turing Model OPTIM

@model function depression_model(bmi, gender, age, vitd, poverty, depression, smoking)
    # Define the length 
    n = min(length(bmi), length(gender), length(age), length(vitd), length(poverty), length(depression), length(smoking))

    # Define the priors
    μ_bmi = 3.388770294648359
    β_age_bmi = 0.01788557065764921
    β_smoking_bmi = -0.39087988490863035
    σ_bmi = 5.004915977740798e-7
    α_age_1 = 32.34836216678736
    α_age_2 = 50.39975492468267
    α_vitd = 4.1886368315939855
    β_smoking_vitd = 0.2919166091445788
    β_age_vitd = 0.20952442146099187
    β_gender_vitd = 0.38303301967633
    β_bmi_vitd = -0.49362778865976903
    σ_vitd = 2.874584294906957e-9
    α_depression = -0.00038446657294942616
    β_vitd_depression = -0.7985800051272052
    β_bmi_depression = -0.772693306645509
    β_age_depression_1 = -0.5726394413036536
    β_age_depression_2 = -0.7768720457539819
    β_age_depression_3 = -1.2345663663139606
    β_poverty_depression = 0.07791503361795347
    β_gender_depression = -0.7461416025159396
    β_smoking_depression = 0.6407812673236092
    
    for i in 1:n
        dist1_age = Normal(α_age_1, 7.2)
        dist2_age = Normal(α_age_2, 8.7)
        dist1_pov = Normal(1, 2)
        dist2_pov = Normal(5, 0.001)
        gender[i] ~ Bernoulli(0.4)
        age[i] ~ MixtureModel([dist1_age, dist2_age], [0.4, 0.6])
        smoking[i] ~ Categorical([0.35, 0.12, 0.53]) 
        poverty[i] ~ MixtureModel([dist1_pov, dist2_pov], [0.5, 0.5])

        bmi[i] ~ LogNormal(μ_bmi + β_age_bmi * age[i] + β_smoking_bmi * smoking[i], σ_bmi)

        vitd[i] ~ LogNormal(α_vitd + β_smoking_vitd * smoking[i] + β_age_vitd * age[i]
        + β_gender_vitd * gender[i] + β_bmi_vitd * bmi[i], σ_vitd)

        if age[i] < 30
            linear_predictor = α_depression + vitd[i] * β_vitd_depression +
            bmi[i] * β_bmi_depression + age[i] * β_age_depression_1 +
            poverty[i] * β_poverty_depression + gender[i] * β_gender_depression +
            smoking[i] * β_smoking_depression

        elseif age[i] >= 30 && age[i] < 50

            linear_predictor = α_depression + vitd[i] * β_vitd_depression +
            bmi[i] * β_bmi_depression + age[i] * β_age_depression_2 +
            poverty[i] * β_poverty_depression + gender[i] * β_gender_depression +
            smoking[i] * β_smoking_depression
        
        else

            linear_predictor = α_depression + vitd[i] * β_vitd_depression +
            bmi[i] * β_bmi_depression + age[i] * β_age_depression_3 +
            poverty[i] * β_poverty_depression + gender[i] * β_gender_depression +
            smoking[i] * β_smoking_depression
        end

        # depression[i] ~ Categorical(softmax([0, linear_predictor, linear_predictor + θ_1, linear_predictor + θ_2]))
        p_depression = logistic(linear_predictor)
        depression[i] ~ Bernoulli(p_depression)

    end
end


depression_model (generic function with 2 methods)

In [28]:
model_depr = depression_model(bmi, gender, age, vitd, poverty_array, repeat([missing], length(bmi)), smoking)
chain_depr = sample(model_depr, PG(10), MCMCThreads(), 100, 6)

Sampling (6 threads)   0%|                              |  ETA: N/A
Sampling (6 threads)  17%|█████                         |  ETA: 0:56:35
Sampling (6 threads)  33%|██████████                    |  ETA: 0:22:38
Sampling (6 threads)  50%|███████████████               |  ETA: 0:11:19
Sampling (6 threads)  67%|████████████████████          |  ETA: 0:05:40
Sampling (6 threads)  83%|█████████████████████████     |  ETA: 0:02:16
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:27:27
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:27:27


Chains MCMC chain (100×1630×6 Array{Float64, 3}):

Iterations        = 1:1:100
Number of chains  = 6
Samples per chain = 100
Wall duration     = 1644.5 seconds
Compute duration  = 5019.05 seconds
parameters        = depression[1], depression[2], depression[3], depression[4], depression[5], depression[6], depression[7], depression[8], depression[9], depression[10], depression[11], depression[12], depression[13], depression[14], depression[15], depression[16], depression[17], depression[18], depression[19], depression[20], depression[21], depression[22], depression[23], depression[24], depression[25], depression[26], depression[27], depression[28], depression[29], depression[30], depression[31], depression[32], depression[33], depression[34], depression[35], depression[36], depression[37], depression[38], depression[39], depression[40], depression[41], depression[42], depression[43], depression[44], depression[45], depression[46], depression[47], depression[48], depression[49], depressio

In [32]:
depression_days = Array{Float64}(undef, length(depression))

for i in 1:length(depression)
    depression_days[i] = mean(chain_depr["depression[$i]"])
end

depression_days

1628-element Vector{Float64}:
 0.09166666666666666
 0.105
 0.035
 0.215
 0.27166666666666667
 0.18333333333333332
 0.2633333333333333
 0.045
 0.22166666666666668
 0.04666666666666667
 ⋮
 0.16833333333333333
 0.09666666666666666
 0.22
 0.07833333333333334
 0.06333333333333334
 0.04666666666666667
 0.058333333333333334
 0.11333333333333333
 0.06666666666666667

In [30]:
total = 0
for i in 1:length(depression)
    is_correct = depression_days[i] == depression[i]
    total += is_correct
end
good_rat = total/ length(depression)

0.8925061425061425

In [26]:
intervention_data = filter(row -> (row."DPQ020" == 3 || row."DPQ020" == 4) && row."LBXVIDMS" < quantile(data."LBXVIDMS", 0.3), data)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Int64,Float64,Float64,Float64,Int64,Int64
1,45.3,1,44.0,42.8,1.05,3,1
2,30.5,0,68.0,40.8,0.35,4,1
3,29.4,1,46.0,32.1,0.92,4,3
4,30.4,0,35.0,36.5,0.23,4,1
5,27.9,1,22.0,43.7,3.03,3,3
6,34.1,0,55.0,43.1,1.0,3,1
7,35.3,1,56.0,32.5,2.04,3,1
8,34.0,0,63.0,48.4,0.8,3,3
9,27.3,0,54.0,47.1,4.79,3,1


In [27]:
#= Convert categorical data to integers =#

intervention_data[!, :"RIAGENDR"] = convert(Array{Int}, intervention_data[!, :"RIAGENDR"])
intervention_data[!, :"DPQ020"] = convert(Array{Int}, intervention_data[!, :"DPQ020"])
intervention_data[!, :"SMQ040"] = convert(Array{Int}, intervention_data[!, :"SMQ040"]);

# Get variables
bmi_interv = intervention_data[!, :"BMXBMI"]
gender_interv = intervention_data[!, :"RIAGENDR"]
age_interv = intervention_data[!, :"RIDAGEYR"]
vitd_interv = intervention_data[!, :"LBXVIDMS"]
poverty_interv = intervention_data[!, :"INDFMMPI"]
depression_interv = intervention_data[!, :"DPQ020"]
smoking_interv = intervention_data[!, :"SMQ040"]
poverty_array_interv = convert(Vector{Float64}, poverty_interv);

In [28]:
model_depr_interv = depression_model(bmi_interv, gender_interv, age_interv, vitd_interv .+ (vitd_interv .* 0.01), poverty_array_interv, repeat([missing], length(bmi)), smoking_interv)
chain_depr_interv = sample(model_depr_interv, PG(10), 100)

Sampling   0%|                                          |  ETA: N/A
Sampling   1%|▍                                         |  ETA: 0:00:38
Sampling   2%|▉                                         |  ETA: 0:00:37
Sampling   3%|█▎                                        |  ETA: 0:00:37
Sampling   4%|█▋                                        |  ETA: 0:00:36
Sampling   5%|██▏                                       |  ETA: 0:00:40
Sampling   6%|██▌                                       |  ETA: 0:00:39
Sampling   7%|███                                       |  ETA: 0:00:38
Sampling   8%|███▍                                      |  ETA: 0:00:37
Sampling   9%|███▊                                      |  ETA: 0:00:37
Sampling  10%|████▎                                     |  ETA: 0:00:36
Sampling  11%|████▋                                     |  ETA: 0:00:36
Sampling  12%|█████                                     |  ETA: 0:00:35
Sampling  13%|█████▌                                    |  ETA: 0:00

Chains MCMC chain (100×56×1 Array{Float64, 3}):

Log evidence      = -1.2859564523026404e12
Iterations        = 1:1:100
Number of chains  = 1
Samples per chain = 100
Wall duration     = 38.93 seconds
Compute duration  = 38.93 seconds
parameters        = depression[1], depression[2], depression[3], depression[4], depression[5], depression[6], depression[7], depression[8], depression[9], depression[10], depression[11], depression[12], depression[13], depression[14], depression[15], depression[16], depression[17], depression[18], depression[19], depression[20], depression[21], depression[22], depression[23], depression[24], depression[25], depression[26], depression[27], depression[28], depression[29], depression[30], depression[31], depression[32], depression[33], depression[34], depression[35], depression[36], depression[37], depression[38], depression[39], depression[40], depression[41], depression[42], depression[43], depression[44], depression[45], depression[46], depression[47], dep

In [29]:
depression_days_interv = Array{Int}(undef, length(depression_interv))

for i in 1:length(depression_interv)
    depression_days_interv[i] = round(mean(chain_depr_interv["depression[$i]"]))
end

depression_days_interv = clamp.(depression_days_interv, 1, 4)

countmap(depression_days_interv)

Dict{Int64, Int64} with 2 entries:
  2 => 21
  3 => 33

In [30]:
countmap(depression_interv)

Dict{Int64, Int64} with 2 entries:
  4 => 20
  3 => 34

In [31]:
total = 0
for i in 1:length(depression_interv)
    is_better = depression_days_interv[i] < depression_interv[i]
    total += is_better
end
good_rat = total/ length(depression_interv)

0.6296296296296297

In [32]:
counterfact_data = filter(row -> (row."DPQ020" == 3 || row."DPQ020" == 4) && row."LBXVIDMS" > quantile(data."LBXVIDMS", 0.7), data)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Int64,Float64,Float64,Float64,Int64,Int64
1,28.2,0,69.0,94.5,1.01,3,3
2,23.9,1,67.0,107.0,1.35,4,1
3,30.4,1,72.0,126.0,4.12,4,3
4,19.1,1,47.0,85.4,0.99,4,1
5,27.5,1,54.0,117.0,1.02,4,1
6,24.5,1,66.0,88.6,1.25,3,3
7,45.0,1,68.0,77.1,1.18,4,3
8,36.8,0,59.0,90.0,1.86,3,1
9,27.1,1,48.0,86.7,4.16,3,1


In [33]:
# Convert categorical data to integers
counterfact_data[!, :"RIAGENDR"] = convert(Array{Int}, counterfact_data[!, :"RIAGENDR"])
counterfact_data[!, :"DPQ020"] = convert(Array{Int}, counterfact_data[!, :"DPQ020"])
counterfact_data[!, :"SMQ040"] = convert(Array{Int}, counterfact_data[!, :"SMQ040"])

# Get variables for counterfactual analysis
bmi_counterfact = counterfact_data[!, :"BMXBMI"]
gender_counterfact = counterfact_data[!, :"RIAGENDR"]
age_counterfact = counterfact_data[!, :"RIDAGEYR"]
vitd_counterfact = counterfact_data[!, :"LBXVIDMS"]
poverty_counterfact = counterfact_data[!, :"INDFMMPI"]
depression_counterfact = counterfact_data[!, :"DPQ020"]
smoking_counterfact = counterfact_data[!, :"SMQ040"]
poverty_array_counterfact = convert(Vector{Float64}, poverty_counterfact);

In [34]:
counterfact_model_depr = depression_model(bmi_counterfact, gender_counterfact, age_counterfact, vitd_counterfact .+ (vitd_counterfact .* 0.01), poverty_array_counterfact, repeat([missing], length(bmi)), smoking_counterfact)
counterfact_chain_depr = sample(counterfact_model_depr, PG(100), 100)

Sampling   0%|                                          |  ETA: N/A
Sampling   1%|▍                                         |  ETA: 0:05:47
Sampling   2%|▉                                         |  ETA: 0:05:38
Sampling   3%|█▎                                        |  ETA: 0:05:30
Sampling   4%|█▋                                        |  ETA: 0:05:28
Sampling   5%|██▏                                       |  ETA: 0:05:25
Sampling   6%|██▌                                       |  ETA: 0:05:23
Sampling   7%|███                                       |  ETA: 0:05:20
Sampling   8%|███▍                                      |  ETA: 0:05:15
Sampling   9%|███▊                                      |  ETA: 0:05:12
Sampling  10%|████▎                                     |  ETA: 0:05:08
Sampling  11%|████▋                                     |  ETA: 0:05:06
Sampling  12%|█████                                     |  ETA: 0:05:02
Sampling  13%|█████▌                                    |  ETA: 0:04

Chains MCMC chain (100×49×1 Array{Float64, 3}):

Log evidence      = -3.9999599247463354e11
Iterations        = 1:1:100
Number of chains  = 1
Samples per chain = 100
Wall duration     = 344.0 seconds
Compute duration  = 344.0 seconds
parameters        = depression[1], depression[2], depression[3], depression[4], depression[5], depression[6], depression[7], depression[8], depression[9], depression[10], depression[11], depression[12], depression[13], depression[14], depression[15], depression[16], depression[17], depression[18], depression[19], depression[20], depression[21], depression[22], depression[23], depression[24], depression[25], depression[26], depression[27], depression[28], depression[29], depression[30], depression[31], depression[32], depression[33], depression[34], depression[35], depression[36], depression[37], depression[38], depression[39], depression[40], depression[41], depression[42], depression[43], depression[44], depression[45], depression[46], depression[47]
inte

In [35]:
depression_days_counterfact = Array{Int}(undef, length(depression_counterfact))

for i in 1:length(depression_counterfact)
    depression_days_interv[i] = round(mean(counterfact_chain_depr["depression[$i]"]))
end

depression_days_counterfact = clamp.(depression_days_counterfact, 1, 4)

countmap(depression_days_counterfact)

Dict{Int64, Int64} with 2 entries:
  4 => 45
  1 => 2

In [36]:
total = 0
for i in 1:length(depression_counterfact)
    is_better = depression_days_counterfact[i] < depression_counterfact[i]
    total += is_better
end
good_rat = total/ length(depression_counterfact)

0.0425531914893617